**Step 1: Load matrix artifacts and verify shapes**

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

# If you are running from inside viraj/notebooks, adjust to project root
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parents[1]

ARTIFACT_DIR = ROOT / "artifacts"
MATRIX_DIR = ARTIFACT_DIR / "matrix"

train_df = pd.read_csv(MATRIX_DIR / "train_df.csv")
test_df = pd.read_csv(MATRIX_DIR / "test_df.csv")
mf_predictions = pd.read_csv(MATRIX_DIR / "mf_predictions_test.csv")
products = pd.read_csv(MATRIX_DIR / "skincare_products.csv")
user_history = pd.read_csv(MATRIX_DIR / "user_history.csv")

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("mf_predictions:", mf_predictions.shape)
print("products:", products.shape)
print("user_history:", user_history.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nMF prediction columns:")
print(mf_predictions.columns.tolist())

print("\nProducts columns:")
print(products.columns.tolist())

train_df: (121096, 3)
test_df: (27822, 3)
mf_predictions: (55640, 23)
products: (2420, 27)
user_history: (9246, 5)

Train columns:
['author_id', 'product_id', 'rating']

MF prediction columns:
['author_id', 'product_id', 'true_rating', 'baseline_score', 'legacy_mf_score', 'item_knn_score', 'metadata_content_score', 'user_rating_count', 'product_rating_count', 'product_avg_rating', 'pred_mean', 'pred_std', 'pred_min', 'pred_max', 'pred_range', 'source_split', 'ridge_ensemble_score', 'mf_score', 'gb_ensemble_score', 'rf_ensemble_score', 'component_score_mean', 'component_score_std', 'component_score_range']

Products columns:
['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'avg_product_rating', 'num_reviews', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category

**Step 2: Build product text for transformer embeddings**

In [2]:
import ast
import re

# Keep IDs consistent for all future joins
train_df["author_id"] = train_df["author_id"].astype(str)
train_df["product_id"] = train_df["product_id"].astype(str)

test_df["author_id"] = test_df["author_id"].astype(str)
test_df["product_id"] = test_df["product_id"].astype(str)

mf_predictions["author_id"] = mf_predictions["author_id"].astype(str)
mf_predictions["product_id"] = mf_predictions["product_id"].astype(str)

products["product_id"] = products["product_id"].astype(str)


def parse_listish(value):
    """
    Convert strings like "['Clean at Sephora', 'Vegan']"
    into readable text.
    """
    if isinstance(value, list):
        return ", ".join(str(x) for x in value)

    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return ", ".join(str(x) for x in parsed)
        except Exception:
            pass

    return text


def clean_text(value):
    text = parse_listish(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


text_columns = [
    "product_name",
    "brand_name",
    "secondary_category",
    "tertiary_category",
    "highlights",
    "ingredients",
]

for col in text_columns:
    products[col] = products[col].apply(clean_text)


def build_product_text(row):
    parts = [
        f"Product: {row['product_name']}",
        f"Brand: {row['brand_name']}",
        f"Category: {row['secondary_category']} {row['tertiary_category']}",
        f"Highlights: {row['highlights']}",
        f"Ingredients: {row['ingredients']}",
    ]

    return " | ".join([p for p in parts if p.strip() and not p.endswith(": ")])


products["product_text"] = products.apply(build_product_text, axis=1)
products["product_text_length"] = products["product_text"].str.len()
products["has_ingredients"] = products["ingredients"].str.len() > 0

print("Products:", products.shape)
print("Products with ingredients:", products["has_ingredients"].sum())
print("Products without ingredients:", (~products["has_ingredients"]).sum())

print("\nProduct text length summary:")
print(products["product_text_length"].describe())

products[[
    "product_id",
    "product_name",
    "brand_name",
    "secondary_category",
    "tertiary_category",
    "has_ingredients",
    "product_text_length",
    "product_text",
]].head(3)


Products: (2420, 30)
Products with ingredients: 2286
Products without ingredients: 134

Product text length summary:
count     2420.000000
mean      1045.658264
std        721.587525
min         76.000000
25%        651.000000
50%        934.000000
75%       1244.000000
max      10475.000000
Name: product_text_length, dtype: float64


,product_id,product_name,brand_name,secondary_category,tertiary_category,has_ingredients,product_text_length,product_text
0,P439055,GENIUS Sleeping Collagen Moisturizer,Algenist,Moisturizers,Moisturizers,True,1172,Product: GENIUS Sleeping Collagen Moisturizer ...
1,P421277,GENIUS Liquid Collagen Serum,Algenist,Treatments,Face Serums,True,1022,Product: GENIUS Liquid Collagen Serum | Brand:...
2,P467602,Triple Algae Eye Renewal Balm Eye Cream,Algenist,Eye Care,Eye Creams & Treatments,True,1630,Product: Triple Algae Eye Renewal Balm Eye Cre...


**Step 2.5: Create optimized transformer text**

In [3]:
def shorten_text(text, max_chars=2500):
    """
    Keep product/category/highlights/ingredients text from becoming too long.
    all-MiniLM-L6-v2 will truncate internally anyway, so we cap text deliberately.
    """
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars]


products["transformer_text"] = products["product_text"].apply(shorten_text)
products["transformer_text_length"] = products["transformer_text"].str.len()

print("Transformer text length summary:")
print(products["transformer_text_length"].describe())

print("\nNumber of capped products:")
print((products["product_text_length"] > products["transformer_text_length"]).sum())

products[[
    "product_id",
    "product_name",
    "product_text_length",
    "transformer_text_length",
    "transformer_text",
]].sort_values("product_text_length", ascending=False).head(5)


Transformer text length summary:
count    2420.000000
mean     1001.803306
std       536.809558
min        76.000000
25%       651.000000
50%       934.000000
75%      1244.000000
max      2500.000000
Name: transformer_text_length, dtype: float64

Number of capped products:
93


,product_id,product_name,product_text_length,transformer_text_length,transformer_text
2415,P501474,The Youth Vault: 13-Piece Vegan Skincare + App...,10475,2500,Product: The Youth Vault: 13-Piece Vegan Skinc...
2087,P483495,Wake Up With Me Morning Routine Kit,7089,2500,Product: Wake Up With Me Morning Routine Kit |...
109,P504888,Aqua Bomb Protect & Glow Travel Kit,6901,2500,Product: Aqua Bomb Protect & Glow Travel Kit |...
2367,P475200,Your Best Skin at Every Age Firming & Smoothin...,6081,2500,Product: Your Best Skin at Every Age Firming &...
903,P504171,Fruit Babies Bestsellers Kit,5225,2500,Product: Fruit Babies Bestsellers Kit | Brand:...


**Step 3: Generate or load transformer embeddings**

In [4]:
from sentence_transformers import SentenceTransformer
from pathlib import Path

TRANSFORMER_DIR = ARTIFACT_DIR / "transformer"
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_PATH = TRANSFORMER_DIR / "viraj_step_product_embeddings.npz"

product_ids = products["product_id"].to_numpy()
texts = products["transformer_text"].tolist()

if EMBEDDING_PATH.exists():
    saved = np.load(EMBEDDING_PATH, allow_pickle=True)
    saved_product_ids = saved["product_ids"].astype(str)

    if np.array_equal(saved_product_ids, product_ids):
        embeddings = saved["embeddings"].astype("float32")
        print("Loaded cached embeddings:", embeddings.shape)
    else:
        print("Cached embeddings do not match product order. Recomputing...")
        model = SentenceTransformer(MODEL_NAME)
        embeddings = model.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        ).astype("float32")

        np.savez_compressed(
            EMBEDDING_PATH,
            product_ids=product_ids,
            embeddings=embeddings,
            model_name=MODEL_NAME,
        )
        print("Saved embeddings:", embeddings.shape)
else:
    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype("float32")

    np.savez_compressed(
        EMBEDDING_PATH,
        product_ids=product_ids,
        embeddings=embeddings,
        model_name=MODEL_NAME,
    )
    print("Saved embeddings:", embeddings.shape)

print("Embedding dtype:", embeddings.dtype)
print("Embedding shape:", embeddings.shape)
print("Embedding min/max:", embeddings.min(), embeddings.max())

# sanity check: normalized embeddings should have norm close to 1
norms = np.linalg.norm(embeddings, axis=1)
print("Norm summary:")
print(pd.Series(norms).describe())


/Users/veerr_89/.venvs/global/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded cached embeddings: (2420, 384)
Embedding dtype: float32
Embedding shape: (2420, 384)
Embedding min/max: -0.21148768 0.2122825
Norm summary:
count    2.420000e+03
mean     1.000000e+00
std      3.048926e-08
min      9.999999e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


**Step 4: Product-to-product similarity sanity check**

In [5]:
from sklearn.neighbors import NearestNeighbors

product_ids = products["product_id"].to_numpy()
product_id_to_idx = {pid: idx for idx, pid in enumerate(product_ids)}

nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn.fit(embeddings)


def similar_products(product_id, top_n=10):
    product_id = str(product_id)

    if product_id not in product_id_to_idx:
        raise ValueError(f"Product id not found: {product_id}")

    idx = product_id_to_idx[product_id]

    distances, indices = nn.kneighbors(
        embeddings[idx].reshape(1, -1),
        n_neighbors=top_n + 1
    )

    rows = []
    for distance, neighbor_idx in zip(distances[0], indices[0]):
        neighbor_id = product_ids[neighbor_idx]

        # skip itself
        if neighbor_id == product_id:
            continue

        item = products.iloc[neighbor_idx]

        rows.append({
            "product_id": neighbor_id,
            "similarity": 1 - float(distance),
            "product_name": item["product_name"],
            "brand_name": item["brand_name"],
            "secondary_category": item["secondary_category"],
            "tertiary_category": item["tertiary_category"],
            "price_usd": item["price_usd"],
            "avg_product_rating": item["avg_product_rating"],
        })

    return pd.DataFrame(rows).head(top_n)


def search_products(term, top_n=10):
    mask = products["product_text"].str.contains(
        term,
        case=False,
        na=False,
        regex=False
    )

    return products.loc[
        mask,
        [
            "product_id",
            "product_name",
            "brand_name",
            "secondary_category",
            "tertiary_category",
            "avg_product_rating",
        ]
    ].head(top_n)


print("Search: moisturizer")
display(search_products("moisturizer", top_n=5))

print("Search: acne")
display(search_products("acne", top_n=5))

print("Search: spf")
display(search_products("spf", top_n=5))


Search: moisturizer


,product_id,product_name,brand_name,secondary_category,tertiary_category,avg_product_rating
0,P439055,GENIUS Sleeping Collagen Moisturizer,Algenist,Moisturizers,Moisturizers,4.5413
5,P384537,GENIUS Ultimate Anti-Aging Cream,Algenist,Moisturizers,Moisturizers,4.2525
9,P379907,Advanced Anti-Aging Repairing Oil,Algenist,Moisturizers,Face Oils,4.4531
10,P296415,Overnight Restorative Cream,Algenist,Moisturizers,Moisturizers,4.3808
12,P282935,Regenerative Anti-Aging Moisturizer,Algenist,Moisturizers,Moisturizers,4.2715


Search: acne


,product_id,product_name,brand_name,secondary_category,tertiary_category,avg_product_rating
18,P442859,ALIVE Prebiotic Balancing Mask,Algenist,Masks,Face Masks,4.3729
40,P442545,Triple Action Cleanser,Alpha-H,Cleansers,Face Wash & Cleansers,4.2500
42,P465741,Wild Huckleberry 8-Acid Polishing Peel Mask,alpyn beauty,Masks,Face Masks,4.6801
50,P505338,Pore Perfecting Liquid Exfoliator with 2% BHA ...,alpyn beauty,Cleansers,Exfoliators,4.9231
69,P455623,The ZenBubble Gel Cream,BeautyBio,Moisturizers,Moisturizers,4.4063


Search: spf


,product_id,product_name,brand_name,secondary_category,tertiary_category,avg_product_rating
4,P311143,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,Algenist,Sunscreen,Face Sunscreen,4.4134
28,P442860,ALIVE Prebiotic Balancing Moisturizer SPF 15,Algenist,Moisturizers,Moisturizers,3.2000
107,P500019,The True Cream - Aqua Bomb Sunscreen Broad Spe...,belif,Sunscreen,Face Sunscreen,4.5248
109,P504888,Aqua Bomb Protect & Glow Travel Kit,belif,Value & Gift Sets,,3.5000
123,P456410,Squalane + Zinc Sheer Mineral Sunscreen SPF 30...,Biossance,Sunscreen,Face Sunscreen,3.8812


**Step 4.1: Similarity checks for 3 query products**

In [6]:
query_ids = {
    "moisturizer": "P439055",
    "acne_bha_exfoliator": "P505338",
    "spf": "P311143",
}

for label, query_product_id in query_ids.items():
    print("=" * 90)
    print(label.upper())

    query = products.loc[
        products["product_id"] == query_product_id,
        [
            "product_id",
            "product_name",
            "brand_name",
            "secondary_category",
            "tertiary_category",
            "avg_product_rating",
        ]
    ]

    print("\nQuery product:")
    display(query)

    print("\nMost similar products:")
    display(similar_products(query_product_id, top_n=10))


MOISTURIZER

Query product:


,product_id,product_name,brand_name,secondary_category,tertiary_category,avg_product_rating
0,P439055,GENIUS Sleeping Collagen Moisturizer,Algenist,Moisturizers,Moisturizers,4.5413



Most similar products:


,product_id,similarity,product_name,brand_name,secondary_category,tertiary_category,price_usd,avg_product_rating
0,P453818,0.910524,GENIUS Collagen Calming Relief,Algenist,Moisturizers,Moisturizers,58.0,4.4640
1,P421277,0.896486,GENIUS Liquid Collagen Serum,Algenist,Treatments,Face Serums,115.0,4.0259
2,P456990,0.872045,Mini GENIUS Liquid Collagen,Algenist,Mini Size,,25.0,3.4412
3,P384537,0.862247,GENIUS Ultimate Anti-Aging Cream,Algenist,Moisturizers,Moisturizers,112.0,4.2525
4,P443358,0.859986,Intensive Moisture Balance Moisturizer,Dermalogica,Moisturizers,Moisturizers,47.0,4.4667
5,P500420,0.859592,The Light Cream,Augustinus Bader,Moisturizers,Moisturizers,180.0,4.3196
6,P500716,0.857516,10 Day Results Kit,Algenist,Value & Gift Sets,,88.0,4.8023
7,P466153,0.855567,Cloud Dew Gel Cream Moisturizer,Summer Fridays,Moisturizers,Moisturizers,45.0,4.4613
8,P432045,0.855386,GENIUS Liquid Collagen Lip Treatment,Algenist,Lip Balms & Treatments,,29.0,3.8721
9,P442839,0.854943,Barrier+ Triple Lipid-Peptide Lotion Moisturizer,Skinfix,Moisturizers,Moisturizers,42.0,4.6503


ACNE_BHA_EXFOLIATOR

Query product:


...

,product_id,similarity,product_name,brand_name,secondary_category,tertiary_category,price_usd,avg_product_rating
0,P480174,0.874566,Waves of Clarity Skincare Set,Herbivore,Value & Gift Sets,,32.0,4.5000
1,P456213,0.868853,GOOPGLOW Microderm Instant Glow Exfoliator,goop,Cleansers,Exfoliators,125.0,4.4821
2,P232915,0.866052,ExfoliKate Intensive Pore Exfoliating Treatment,Kate Somerville,Cleansers,Exfoliators,98.0,4.4375
3,P397310,0.864635,PoreDermabrasion Pore Perfecting Exfoliator,Dr. Brandt Skincare,Cleansers,Exfoliators,59.0,4.2437
4,P503872,0.861855,Mini Mandelic Acid + Superfood Unity Exfoliant,Youth To The People,Cleansers,Toners,16.0,4.8000
5,P475630,0.861855,Mini Mandelic Acid + Superfood Unity Exfoliant,Youth To The People,Cleansers,Toners,16.0,4.6747
6,P505379,0.854235,Multi Action Clear Gentle Daily Clarifying Cle...,StriVectin,Cleansers,Face Wash & Cleansers,29.0,NaN
7,P474326,0.851155,Pink Cloud Soft Moisture Cream,Herbivore,Moisturizers,Moisturizers,46.0,3.7860
8,P412407,0.851038,Clarifying Moisturizer,Tata Harper,Moisturizers,Moisturizers,130.0,3.8519
9,P470243,0.850604,PRO Strength Microdermabrasion Blackhead Elimi...,Peter Thomas Roth,Cleansers,Exfoliators,68.0,4.4936


SPF

Query product:


,product_id,product_name,brand_name,secondary_category,tertiary_category,avg_product_rating
4,P311143,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,Algenist,Sunscreen,Face Sunscreen,4.4134



Most similar products:


,product_id,similarity,product_name,brand_name,secondary_category,tertiary_category,price_usd,avg_product_rating
0,P417118,0.888129,The Broad Spectrum SPF 50 UV Protecting Fluid,La Mer,Sunscreen,Face Sunscreen,105.0,4.2537
1,P470057,0.883678,Mineral Sheerscreen Sunscreen SPF 30 PA+++,Supergoop!,Sunscreen,Face Sunscreen,38.0,3.9473
2,P469524,0.883544,RESIST Super-Light Daily Wrinkle Defense SPF 30,Paula's Choice,Sunscreen,Face Sunscreen,37.0,4.2556
3,P482322,0.881461,Mini Mineral Mattescreen Sunscreen SPF 40 PA+++,Supergoop!,Sunscreen,Face Sunscreen,22.0,4.1250
4,P482321,0.880904,Mini Mineral Sheerscreen Sunscreen SPF 30 PA+++,Supergoop!,Sunscreen,Face Sunscreen,22.0,3.9473
5,P476733,0.878618,Mineral Mattescreen Sunscreen SPF 40 PA+++,Supergoop!,Sunscreen,Face Sunscreen,38.0,4.0153
6,P500993,0.875978,"The Sunscreen - 100% Mineral, Broad Spectrum S...",Nécessaire,Sunscreen,Face Sunscreen,25.0,4.5322
7,P456392,0.875775,Daily UV Defense Invisible Broad Spectrum SPF ...,innisfree,Sunscreen,Face Sunscreen,16.0,4.5393
8,P479646,0.875106,Mini City Skin Age Defense Broad Spectrum SPF ...,Murad,Sunscreen,Face Sunscreen,25.0,4.3333
9,P417980,0.873635,City Skin Age Defense Broad Spectrum SPF 50 PA...,Murad,Sunscreen,Face Sunscreen,69.0,4.3219


**Step 5: Build user content profiles from liked training products**

In [7]:
LIKED_RATING_THRESHOLD = 4.0

product_id_to_idx = {
    pid: idx for idx, pid in enumerate(products["product_id"].to_numpy())
}

# Keep only train rows whose products exist in transformer catalog
train_with_embeddings = train_df[
    train_df["product_id"].isin(product_id_to_idx)
].copy()

liked_train = train_with_embeddings[
    train_with_embeddings["rating"] >= LIKED_RATING_THRESHOLD
].copy()

print("Train rows with embeddings:", train_with_embeddings.shape)
print("Liked train rows:", liked_train.shape)
print("Users with at least one liked product:", liked_train["author_id"].nunique())
print("Total train users:", train_df["author_id"].nunique())


def build_user_content_profiles(train_df, liked_threshold=4.0):
    user_vectors = {}
    user_profile_rows = []

    for user_id, user_rows in train_df.groupby("author_id"):
        liked_rows = user_rows[user_rows["rating"] >= liked_threshold]

        # Prefer liked products. If none exist, fall back to all rated products.
        source_rows = liked_rows if len(liked_rows) > 0 else user_rows

        product_indices = [
            product_id_to_idx[pid]
            for pid in source_rows["product_id"]
            if pid in product_id_to_idx
        ]

        if len(product_indices) == 0:
            continue

        vector = embeddings[product_indices].mean(axis=0)

        # normalize averaged vector for cosine similarity
        norm = np.linalg.norm(vector)
        if norm > 0:
            vector = vector / norm

        user_vectors[user_id] = vector.astype("float32")

        user_profile_rows.append({
            "author_id": user_id,
            "profile_product_count": len(product_indices),
            "liked_product_count": len(liked_rows),
            "mean_train_rating": user_rows["rating"].mean(),
        })

    user_profile_df = pd.DataFrame(user_profile_rows)

    return user_vectors, user_profile_df


user_content_vectors, user_content_profile = build_user_content_profiles(
    train_with_embeddings,
    liked_threshold=LIKED_RATING_THRESHOLD
)

print("User content vectors:", len(user_content_vectors))
print("User content profile:", user_content_profile.shape)

print("\nProfile count summary:")
print(user_content_profile[[
    "profile_product_count",
    "liked_product_count",
    "mean_train_rating",
]].describe())

user_content_profile.head()

Train rows with embeddings: (121096, 3)
Liked train rows: (105803, 3)
Users with at least one liked product: 9213
Total train users: 9246
User content vectors: 9246
User content profile: (9246, 4)

Profile count summary:
       profile_product_count  liked_product_count  mean_train_rating
count            9246.000000          9246.000000        9246.000000
mean               11.468419            11.443111           4.400571
std                10.294718            10.313546           0.614394
min                 1.000000             0.000000           1.000000
25%                 6.000000             6.000000           4.090909
50%                 8.000000             8.000000           4.571429
75%                12.000000            12.000000           4.900000
max               175.000000           175.000000           5.000000


,author_id,profile_product_count,liked_product_count,mean_train_rating
0,10000770719,9,9,5.000000
1,1000235057,16,16,3.653846
2,10003868106,7,7,4.571429
3,10005368592,3,3,3.000000
4,10005786204,5,5,4.000000


**Step 6: Score held-out test products using transformer content similarity**

In [8]:
# Start from MF predictions so we compare on the exact same test rows
eval_df = mf_predictions.copy()

# Ensure test products exist in transformer catalog
eval_df = eval_df[
    eval_df["product_id"].isin(product_id_to_idx)
].copy()

content_scores = []

for row in eval_df.itertuples(index=False):
    user_id = row.author_id
    product_id = row.product_id

    user_vec = user_content_vectors.get(user_id)
    product_idx = product_id_to_idx.get(product_id)

    if user_vec is None or product_idx is None:
        content_scores.append(np.nan)
    else:
        score = float(np.dot(user_vec, embeddings[product_idx]))
        content_scores.append(score)

eval_df["content_score"] = content_scores

print("Eval rows:", eval_df.shape)
print("Missing content scores:", eval_df["content_score"].isna().sum())

print("\nContent score summary:")
print(eval_df["content_score"].describe())

eval_df.head()


Eval rows: (55640, 24)
Missing content scores: 0

Content score summary:
count    55640.000000
mean         0.809176
std          0.079123
min          0.209516
25%          0.790763
50%          0.829558
75%          0.856503
max          0.994601
Name: content_score, dtype: float64


,author_id,product_id,true_rating,baseline_score,legacy_mf_score,item_knn_score,metadata_content_score,user_rating_count,product_rating_count,product_avg_rating,...,pred_range,source_split,ridge_ensemble_score,mf_score,gb_ensemble_score,rf_ensemble_score,component_score_mean,component_score_std,component_score_range,content_score
0,10000770719,P404338,4.0,5.000000,4.842761,5.000000,5.000000,9,108,4.148148,...,0.157239,validation,4.849577,4.849577,4.822625,4.762637,4.960690,0.078619,0.157239,0.808294
1,10000770719,P454936,5.0,5.000000,4.802196,5.000000,5.000000,9,176,4.113636,...,0.197804,validation,4.823746,4.823746,4.832395,4.805454,4.950549,0.098902,0.197804,0.773950
2,1000235057,P454799,1.0,3.653846,3.875415,1.017612,3.464530,26,163,4.650307,...,2.857802,validation,3.021674,3.021674,2.368018,2.219964,3.002851,1.334102,2.857802,0.543337
3,1000235057,P427414,4.0,3.653846,3.592358,3.233547,3.611173,26,136,4.183824,...,0.420299,validation,3.537052,3.537052,3.579929,3.584698,3.522731,0.194498,0.420299,0.843055
4,1000235057,P416552,3.0,3.653846,3.991763,4.906335,3.713700,26,30,4.600000,...,1.252489,validation,4.302204,4.302204,4.253224,4.162977,4.066411,0.578982,1.252489,0.817662


In [9]:
def content_score_to_rating(score):
    """
    Convert cosine similarity [roughly 0 to 1] into rating scale [1 to 5].
    This is simple and interpretable for a baseline.
    """
    score = np.clip(score, 0, 1)
    return 1 + 4 * score


eval_df["content_rating_score"] = eval_df["content_score"].apply(content_score_to_rating)

print("Content rating score summary:")
print(eval_df["content_rating_score"].describe())

eval_df[[
    "author_id",
    "product_id",
    "true_rating",
    "mf_score",
    "content_score",
    "content_rating_score",
]].head()


Content rating score summary:
count    55640.000000
mean         4.236705
std          0.316491
min          1.838064
25%          4.163052
50%          4.318231
75%          4.426011
max          4.978403
Name: content_rating_score, dtype: float64


,author_id,product_id,true_rating,mf_score,content_score,content_rating_score
0,10000770719,P404338,4.0,4.849577,0.808294,4.233176
1,10000770719,P454936,5.0,4.823746,0.773950,4.095801
2,1000235057,P454799,1.0,3.021674,0.543337,3.173348
3,1000235057,P427414,4.0,3.537052,0.843055,4.372221
4,1000235057,P416552,3.0,4.302204,0.817662,4.270647


**Step 7: Compare MF vs transformer content using RMSE/MAE**

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


mf_rmse = rmse(eval_df["true_rating"], eval_df["mf_score"])
mf_mae = mae(eval_df["true_rating"], eval_df["mf_score"])

content_rmse = rmse(eval_df["true_rating"], eval_df["content_rating_score"])
content_mae = mae(eval_df["true_rating"], eval_df["content_rating_score"])

results = pd.DataFrame({
    "model": [
        "Matrix Factorization",
        "Transformer Content",
    ],
    "RMSE": [
        mf_rmse,
        content_rmse,
    ],
    "MAE": [
        mf_mae,
        content_mae,
    ],
}).sort_values("RMSE")

results


,model,RMSE,MAE
0,Matrix Factorization,0.778599,0.491260
1,Transformer Content,1.005774,0.813879


In [11]:
eval_df["mf_abs_error"] = (eval_df["true_rating"] - eval_df["mf_score"]).abs()
eval_df["content_abs_error"] = (eval_df["true_rating"] - eval_df["content_rating_score"]).abs()

print("Mean absolute error by true rating:")
display(
    eval_df.groupby("true_rating")[[
        "mf_abs_error",
        "content_abs_error",
    ]].mean()
)

print("Cases where transformer is much better than MF:")
display(
    eval_df.assign(
        transformer_advantage=eval_df["mf_abs_error"] - eval_df["content_abs_error"]
    ).sort_values("transformer_advantage", ascending=False)[[
        "author_id",
        "product_id",
        "true_rating",
        "mf_score",
        "content_rating_score",
        "mf_abs_error",
        "content_abs_error",
        "transformer_advantage",
    ]].head(10)
)

print("Cases where MF is much better than transformer:")
display(
    eval_df.assign(
        mf_advantage=eval_df["content_abs_error"] - eval_df["mf_abs_error"]
    ).sort_values("mf_advantage", ascending=False)[[
        "author_id",
        "product_id",
        "true_rating",
        "mf_score",
        "content_rating_score",
        "mf_abs_error",
        "content_abs_error",
        "mf_advantage",
    ]].head(10)
)


Mean absolute error by true rating:


,mf_abs_error,content_abs_error
true_rating,,
1.000000,2.438836,3.098304
1.500000,0.946032,2.564784
2.000000,1.695671,2.128908
2.500000,1.209739,1.384541
3.000000,0.951991,1.160897
3.333333,1.451586,0.984763
3.500000,0.823573,0.709653
4.000000,0.420272,0.343428
4.500000,0.331799,0.270093


Cases where transformer is much better than MF:


,author_id,product_id,true_rating,mf_score,content_rating_score,mf_abs_error,content_abs_error,transformer_advantage
41397,2590642040,P431418,5.0,1.677538,4.190661,3.322462,0.809339,2.513123
29145,10946144511,P397890,2.0,4.920274,2.598692,2.920274,0.598692,2.321582
47228,5566779122,P188309,5.0,1.952390,4.259316,3.047610,0.740684,2.306926
13582,2590642040,P446930,5.0,2.122595,4.233921,2.877405,0.766079,2.111326
41398,2590642040,P3550,4.0,1.686397,4.238022,2.313603,0.238022,2.075581
49272,6186223334,P214002,5.0,2.281544,4.343576,2.718456,0.656424,2.062032
40370,2466210403,P432818,2.0,4.870196,2.824663,2.870196,0.824663,2.045533
14051,26813893920,P432818,1.0,4.475540,2.451375,3.475540,1.451375,2.024165
13583,2590642040,P474078,4.0,1.868348,4.146347,2.131652,0.146347,1.985305
33880,1598253214,P469439,5.0,2.468797,4.452883,2.531203,0.547117,1.984086


Cases where MF is much better than transformer:


,author_id,product_id,true_rating,mf_score,content_rating_score,mf_abs_error,content_abs_error,mf_advantage
46578,5394009495,P454794,1.0,1.195981,4.631846,0.195981,3.631846,3.435865
31630,1306217000,P421998,1.0,1.269143,4.502309,0.269143,3.502309,3.233165
33881,1598253214,P418218,1.0,1.369242,4.584083,0.369242,3.584083,3.214842
24177,7448860052,P467970,5.0,4.916224,1.855598,0.083776,3.144402,3.060626
20231,5808372586,P461933,1.0,1.224305,4.239555,0.224305,3.239555,3.015250
9376,2158566406,P401158,5.0,4.942084,2.001962,0.057916,2.998038,2.940122
7929,1959225222,P467970,5.0,4.890496,1.961206,0.109504,3.038794,2.929290
16579,33037037502,P467970,5.0,4.811097,1.885550,0.188903,3.114450,2.925547
21140,6070053218,P454794,1.0,1.540046,4.451971,0.540046,3.451971,2.911925
40120,2442534239,P401158,5.0,4.872731,1.972806,0.127269,3.027194,2.899925


**Step 8: Build hybrid score**

hybrid_score = 0.7 * mf_score + 0.3 * content_rating_score


In [12]:
ALPHA = 0.7

eval_df["hybrid_score"] = (
    ALPHA * eval_df["mf_score"] +
    (1 - ALPHA) * eval_df["content_rating_score"]
)

hybrid_rmse = rmse(eval_df["true_rating"], eval_df["hybrid_score"])
hybrid_mae = mae(eval_df["true_rating"], eval_df["hybrid_score"])

comparison = pd.DataFrame({
    "model": [
        "Matrix Factorization",
        "Transformer Content",
        f"Hybrid MF {ALPHA:.1f} + Content {1 - ALPHA:.1f}",
    ],
    "RMSE": [
        mf_rmse,
        content_rmse,
        hybrid_rmse,
    ],
    "MAE": [
        mf_mae,
        content_mae,
        hybrid_mae,
    ],
}).sort_values("RMSE")

comparison


,model,RMSE,MAE
0,Matrix Factorization,0.778599,0.491260
2,Hybrid MF 0.7 + Content 0.3,0.802104,0.577116
1,Transformer Content,1.005774,0.813879


In [13]:
alpha_results = []

for alpha in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    score = (
        alpha * eval_df["mf_score"] +
        (1 - alpha) * eval_df["content_rating_score"]
    )

    alpha_results.append({
        "alpha_mf_weight": alpha,
        "content_weight": 1 - alpha,
        "RMSE": rmse(eval_df["true_rating"], score),
        "MAE": mae(eval_df["true_rating"], score),
    })

alpha_results = pd.DataFrame(alpha_results).sort_values("RMSE")
alpha_results


,alpha_mf_weight,content_weight,RMSE,MAE
10,1.0,0.0,0.778599,0.491260
9,0.9,0.1,0.781382,0.518944
8,0.8,0.2,0.789267,0.547549
7,0.7,0.3,0.802104,0.577116
6,0.6,0.4,0.819660,0.607855
5,0.5,0.5,0.841639,0.639752
4,0.4,0.6,0.867707,0.672715
3,0.3,0.7,0.897506,0.706778
2,0.2,0.8,0.930678,0.741761
1,0.1,0.9,0.966876,0.777528


**Step 9: Hit Rate@K for MF, content, and hybrid**

In [14]:
# Build fast lookup: user -> products already seen in training
train_user_items = (
    train_df.groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

# Candidate products are products that appear in MF/item space and transformer catalog
candidate_product_ids = sorted(
    set(train_df["product_id"].unique()) & set(product_id_to_idx.keys())
)

print("Candidate products:", len(candidate_product_ids))


def recommend_by_content(user_id, top_n=20):
    user_vec = user_content_vectors.get(user_id)

    if user_vec is None:
        return []

    already_seen = train_user_items.get(user_id, set())

    candidate_ids = [
        pid for pid in candidate_product_ids
        if pid not in already_seen
    ]

    candidate_indices = [product_id_to_idx[pid] for pid in candidate_ids]
    candidate_embeddings = embeddings[candidate_indices]

    scores = candidate_embeddings @ user_vec

    ranked = pd.DataFrame({
        "product_id": candidate_ids,
        "score": scores,
    }).sort_values("score", ascending=False)

    return ranked.head(top_n)["product_id"].tolist()


def hit_rate_at_k_from_recommender(test_df, recommender_func, k=10):
    hits = 0
    total = 0

    for row in test_df.itertuples(index=False):
        user_id = row.author_id
        true_item = row.product_id

        recs = recommender_func(user_id, top_n=k)

        if len(recs) == 0:
            continue

        hits += int(true_item in recs)
        total += 1

    return hits / total if total > 0 else np.nan


content_hit_rates = {}
for k in [5, 10, 20]:
    hr = hit_rate_at_k_from_recommender(
        test_df,
        recommend_by_content,
        k=k
    )
    content_hit_rates[f"hit_rate_at_{k}"] = float(hr)
    print(f"Content Hit Rate @{k}: {hr:.4f}")


Candidate products: 1890
Content Hit Rate @5: 0.0190
Content Hit Rate @10: 0.0287
Content Hit Rate @20: 0.0434


**Step 10: Load latest matrix metrics and build fresh ranking comparison**

This uses `artifacts/matrix/metrics.json`, not the old handoff metrics.

In [15]:
import json

MATRIX_METRICS_PATH = MATRIX_DIR / "metrics.json"

with open(MATRIX_METRICS_PATH, "r") as f:
    matrix_metrics = json.load(f)

mf_ranking = matrix_metrics.get("ranking", {}).get("mf", {})

ranking_comparison = pd.DataFrame([
    {
        "model": "Matrix Factorization",
        "Hit Rate @5": mf_ranking.get("hit_rate_at_5", np.nan),
        "Hit Rate @10": mf_ranking.get("hit_rate_at_10", np.nan),
        "Hit Rate @20": mf_ranking.get("hit_rate_at_20", np.nan),
        "source": "artifacts/matrix/metrics.json",
    },
    {
        "model": "Transformer Content",
        "Hit Rate @5": content_hit_rates.get("hit_rate_at_5", np.nan),
        "Hit Rate @10": content_hit_rates.get("hit_rate_at_10", np.nan),
        "Hit Rate @20": content_hit_rates.get("hit_rate_at_20", np.nan),
        "source": "current transformer notebook run",
    },
])

ranking_comparison

,model,Hit Rate @5,Hit Rate @10,Hit Rate @20,source
0,Matrix Factorization,NaN,NaN,NaN,artifacts/matrix/metrics.json
1,Transformer Content,0.01905,0.028682,0.043419,current transformer notebook run


In [16]:
mf_hr10 = mf_ranking.get("hit_rate_at_10", np.nan)
content_hr10 = content_hit_rates.get("hit_rate_at_10", np.nan)

print(f"MF Hit Rate@10: {mf_hr10:.4f}")
print(f"Content Hit Rate@10: {content_hr10:.4f}")

if mf_hr10 and not np.isnan(mf_hr10):
    print("\\nImprovement over MF:")
    print(f"Content vs MF: {content_hr10 / mf_hr10:.2f}x")

print("\\nNote: true hybrid Hit Rate@K is not recomputed here because MF candidate-level scores are not exported yet.")

MF Hit Rate@10: nan
Content Hit Rate@10: 0.0287
\nNote: true hybrid Hit Rate@K is not recomputed here because MF candidate-level scores are not exported yet.


**Step 11: Export fresh transformer and Bayesian handoff artifacts**

This overwrites `artifacts/transformer/` and `artifacts/handoff/` using the current matrix split.

In [17]:
HANDOFF_DIR = ARTIFACT_DIR / "handoff"
TRANSFORMER_DIR = ARTIFACT_DIR / "transformer"
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)
TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)

# Save transformer product catalog and embedding metadata for this run.
products.to_csv(TRANSFORMER_DIR / "product_catalog.csv", index=False)
np.savez_compressed(
    TRANSFORMER_DIR / "product_embeddings.npz",
    product_ids=product_ids,
    embeddings=embeddings,
    model_name=MODEL_NAME,
)
(TRANSFORMER_DIR / "embedding_model.txt").write_text(MODEL_NAME + "\n")

# Save product nearest neighbors for product-to-product inspection.
neighbors_rows = []
neighbor_count = 10
for product_idx, product_id in enumerate(product_ids):
    distances, indices = nn.kneighbors(
        embeddings[product_idx].reshape(1, -1),
        n_neighbors=neighbor_count + 1,
    )
    rank = 0
    for distance, neighbor_idx in zip(distances[0], indices[0]):
        neighbor_id = product_ids[neighbor_idx]
        if neighbor_id == product_id:
            continue
        rank += 1
        item = products.iloc[neighbor_idx]
        neighbors_rows.append({
            "product_id": product_id,
            "neighbor_product_id": neighbor_id,
            "rank": rank,
            "similarity": 1 - float(distance),
            "neighbor_product_name": item["product_name"],
            "neighbor_brand_name": item["brand_name"],
            "neighbor_secondary_category": item["secondary_category"],
            "neighbor_tertiary_category": item["tertiary_category"],
        })

product_neighbors = pd.DataFrame(neighbors_rows)
product_neighbors.to_csv(TRANSFORMER_DIR / "product_neighbors.csv", index=False)

print("Saved transformer catalog:", products.shape)
print("Saved product embeddings:", embeddings.shape)
print("Saved product neighbors:", product_neighbors.shape)

Saved transformer catalog: (2420, 32)
Saved product embeddings: (2420, 384)
Saved product neighbors: (24200, 8)


In [18]:
# Build Bayesian handoff features from the current eval_df.
bayesian_handoff_features = eval_df.copy()
bayesian_handoff_features["mf_content_gap"] = (
    bayesian_handoff_features["mf_score"] - bayesian_handoff_features["content_rating_score"]
).abs()

# Attach user history from matrix artifacts.
user_history_export = user_history.copy()
user_history_export["author_id"] = user_history_export["author_id"].astype(str)
bayesian_handoff_features = bayesian_handoff_features.merge(
    user_history_export,
    on="author_id",
    how="left",
)

# Clean duplicate columns caused by merging Ridge ensemble features
# with matrix user_history. BNN expects the canonical unsuffixed name.
if "user_rating_count" not in bayesian_handoff_features.columns:
    if "user_rating_count_y" in bayesian_handoff_features.columns:
        bayesian_handoff_features["user_rating_count"] = bayesian_handoff_features["user_rating_count_y"]
    elif "user_rating_count_x" in bayesian_handoff_features.columns:
        bayesian_handoff_features["user_rating_count"] = bayesian_handoff_features["user_rating_count_x"]
    else:
        raise ValueError("Could not recover user_rating_count after handoff merges.")

drop_duplicate_merge_cols = [
    col for col in [
        "user_rating_count_x",
        "user_rating_count_y",
    ]
    if col in bayesian_handoff_features.columns
]

bayesian_handoff_features = bayesian_handoff_features.drop(
    columns=drop_duplicate_merge_cols
)

print("Canonical user_rating_count column ready:",
      "user_rating_count" in bayesian_handoff_features.columns)


# Attach user content profile summary from this transformer notebook.
bayesian_handoff_features = bayesian_handoff_features.merge(
    user_content_profile,
    on="author_id",
    how="left",
)

# Attach product metadata.
product_meta_cols = [
    "product_id",
    "product_name",
    "brand_name",
    "secondary_category",
    "tertiary_category",
    "price_usd",
    "avg_product_rating",
    "num_reviews",
    "loves_count",
]
existing_product_meta_cols = [c for c in product_meta_cols if c in products.columns]
bayesian_handoff_features = bayesian_handoff_features.merge(
    products[existing_product_meta_cols],
    on="product_id",
    how="left",
)

bayesian_handoff_features.to_csv(
    HANDOFF_DIR / "bayesian_handoff_features.csv",
    index=False,
)

print("Saved Bayesian handoff features:", bayesian_handoff_features.shape)
bayesian_handoff_features.head()

Canonical user_rating_count column ready: True
Saved Bayesian handoff features: (55640, 43)


,author_id,product_id,true_rating,baseline_score,legacy_mf_score,item_knn_score,metadata_content_score,product_rating_count,product_avg_rating,pred_mean,...,liked_product_count,mean_train_rating,product_name,brand_name,secondary_category,tertiary_category,price_usd,avg_product_rating,num_reviews,loves_count
0,10000770719,P404338,4.0,5.000000,4.842761,5.000000,5.000000,108,4.148148,4.960690,...,9,5.000000,Whipped Argan Oil Face Butter,Josie Maran,Moisturizers,Moisturizers,45.0,4.1421,1393.0,40417
1,10000770719,P454936,5.0,5.000000,4.802196,5.000000,5.000000,176,4.113636,4.950549,...,9,5.000000,Mini Hyaluronic Serum,Dr. Barbara Sturm,Mini Size,,110.0,3.9386,879.0,4691
2,1000235057,P454799,1.0,3.653846,3.875415,1.017612,3.464530,163,4.650307,3.002851,...,16,3.653846,Mini SuperYou Daily Stress Management,Moon Juice,Mini Size,,25.0,4.5749,494.0,5313
3,1000235057,P427414,4.0,3.653846,3.592358,3.233547,3.611173,136,4.183824,3.522731,...,16,3.653846,Natural Moisturizing Factors + HA,The Ordinary,Treatments,Face Serums,6.5,4.1472,2052.0,258456
4,1000235057,P416552,3.0,3.653846,3.991763,4.906335,3.713700,30,4.600000,4.066411,...,16,3.653846,POWER Recharging Night Pressed Serum,Algenist,Treatments,Face Serums,95.0,4.3854,205.0,10859


In [19]:
# Save fresh handoff metrics tied to the current matrix split.
content_rmse = rmse(eval_df["true_rating"], eval_df["content_rating_score"])
content_mae = mae(eval_df["true_rating"], eval_df["content_rating_score"])
hybrid_rmse = rmse(eval_df["true_rating"], eval_df["hybrid_score"])
hybrid_mae = mae(eval_df["true_rating"], eval_df["hybrid_score"])

split_report = {
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "mf_prediction_rows": int(len(mf_predictions)),
    "handoff_rows": int(len(bayesian_handoff_features)),
    "train_users": int(train_df["author_id"].nunique()),
    "test_users": int(test_df["author_id"].nunique()),
    "train_products": int(train_df["product_id"].nunique()),
    "test_products": int(test_df["product_id"].nunique()),
    "candidate_products": int(len(candidate_product_ids)),
}

hybrid_metrics = {
    "matrix_metrics_source": str(MATRIX_METRICS_PATH.relative_to(ROOT)),
    "global_mean": matrix_metrics.get("global_mean"),
    "user_item_baseline": matrix_metrics.get("user_item_baseline"),
    "biased_mf": matrix_metrics.get("biased_mf"),
    "content_only": {
        "rmse": float(content_rmse),
        "mae": float(content_mae),
        "evaluated_rows": int(len(eval_df)),
    },
    "hybrid": {
        "alpha": float(ALPHA),
        "rmse": float(hybrid_rmse),
        "mae": float(hybrid_mae),
        "evaluated_rows": int(len(eval_df)),
    },
    "ranking": {
        "mf": mf_ranking,
        "content": {
            **content_hit_rates,
            "evaluated_users": int(test_df["author_id"].nunique()),
        },
        "hybrid": None,
        "note": "Hybrid Hit Rate@K requires candidate-level MF scores; not recomputed in this notebook.",
    },
    "decision_gate": {
        "content_beats_mf_hit_rate_at_10": bool(content_hit_rates.get("hit_rate_at_10", np.nan) > mf_ranking.get("hit_rate_at_10", np.nan)),
        "hybrid_beats_mf_rmse": bool(hybrid_rmse < matrix_metrics.get("biased_mf", {}).get("rmse", np.inf)),
    },
    "split_report": split_report,
}

with open(HANDOFF_DIR / "hybrid_metrics.json", "w") as f:
    json.dump(hybrid_metrics, f, indent=2)

print("Saved fresh hybrid metrics:")
print(json.dumps({
    "split_report": split_report,
    "biased_mf": hybrid_metrics["biased_mf"],
    "content_only": hybrid_metrics["content_only"],
    "hybrid": hybrid_metrics["hybrid"],
    "ranking": hybrid_metrics["ranking"],
    "decision_gate": hybrid_metrics["decision_gate"],
}, indent=2))

Saved fresh hybrid metrics:
{
  "split_report": {
    "train_rows": 121096,
    "test_rows": 27822,
    "mf_prediction_rows": 55640,
    "handoff_rows": 55640,
    "train_users": 9246,
    "test_users": 9246,
    "train_products": 1890,
    "test_products": 1684,
    "candidate_products": 1890
  },
  "biased_mf": {
    "evaluated_rows": 27822,
    "mae": 0.49122524774066767,
    "rmse": 0.7806867907341931
  },
  "content_only": {
    "rmse": 1.0057737111168612,
    "mae": 0.8138789058877871,
    "evaluated_rows": 55640
  },
  "hybrid": {
    "alpha": 0.7,
    "rmse": 0.8021039974326994,
    "mae": 0.5771155883838441,
    "evaluated_rows": 55640
  },
  "ranking": {
    "mf": {},
    "content": {
      "hit_rate_at_5": 0.01904967292071023,
      "hit_rate_at_10": 0.028682337718352383,
      "hit_rate_at_20": 0.04341887714758105,
      "evaluated_users": 9246
    },
    "hybrid": null,
    "note": "Hybrid Hit Rate@K requires candidate-level MF scores; not recomputed in this notebook."
  }

**Step 12: Export validation**

In [20]:
# Validate that handoff rows now match latest matrix MF predictions.
latest_mf = pd.read_csv(MATRIX_DIR / "mf_predictions_test.csv")
latest_handoff = pd.read_csv(HANDOFF_DIR / "bayesian_handoff_features.csv")

latest_mf["author_id"] = latest_mf["author_id"].astype(str)
latest_mf["product_id"] = latest_mf["product_id"].astype(str)
latest_handoff["author_id"] = latest_handoff["author_id"].astype(str)
latest_handoff["product_id"] = latest_handoff["product_id"].astype(str)

alignment = latest_handoff[["author_id", "product_id"]].merge(
    latest_mf[["author_id", "product_id"]],
    on=["author_id", "product_id"],
    how="outer",
    indicator=True,
)

print("MF rows:", len(latest_mf))
print("Handoff rows:", len(latest_handoff))
print("Alignment counts:")
print(alignment["_merge"].value_counts())

assert len(latest_mf) == len(latest_handoff), "Handoff row count must match latest MF predictions."
assert (alignment["_merge"] == "both").all(), "Handoff user-product pairs must match latest MF predictions."

print("Validation passed: handoff artifacts match latest matrix artifacts.")

MF rows: 55640
Handoff rows: 55640
Alignment counts:
_merge
both          55640
left_only         0
right_only        0
Name: count, dtype: int64
Validation passed: handoff artifacts match latest matrix artifacts.


/var/folders/57/vnd70s_n33xdwqkg8pdrndwc0000gn/T/ipykernel_27775/1662656352.py:3: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  latest_handoff = pd.read_csv(HANDOFF_DIR / "bayesian_handoff_features.csv")


In [21]:
from pathlib import Path
import json
import shutil
import pandas as pd

CURRENT_RUN_PATH = ARTIFACT_DIR / "current_run.json"

if not CURRENT_RUN_PATH.exists():
    raise FileNotFoundError("Run the final export cell in Ishita's ensemble notebook first.")

current_run = json.loads(CURRENT_RUN_PATH.read_text())
RUN_ID = current_run["run_id"]
RUN_DIR = Path(current_run["run_dir"])

STEP2_DIR = RUN_DIR / "step2_transformer"
STEP2_TRANSFORMER_DIR = STEP2_DIR / "transformer"
STEP2_HANDOFF_DIR = STEP2_DIR / "handoff"

STEP2_TRANSFORMER_DIR.mkdir(parents=True, exist_ok=True)
STEP2_HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

copied = []

for name in [
    "product_catalog.csv",
    "product_embeddings.npz",
    "viraj_step_product_embeddings.npz",
    "embedding_model.txt",
    "product_neighbors.csv",
    "qualitative_checks.csv",
]:
    src = TRANSFORMER_DIR / name
    if src.exists():
        dst = STEP2_TRANSFORMER_DIR / name
        shutil.copy2(src, dst)
        copied.append(dst)

for name in [
    "bayesian_handoff_features.csv",
    "hybrid_metrics.json",
]:
    src = HANDOFF_DIR / name
    if src.exists():
        dst = STEP2_HANDOFF_DIR / name
        shutil.copy2(src, dst)
        copied.append(dst)

current_run["step2_transformer_dir"] = str(STEP2_DIR)
current_run["stage"] = "step2_transformer_complete"
CURRENT_RUN_PATH.write_text(json.dumps(current_run, indent=2, sort_keys=True) + "\n")

display(pd.DataFrame({
    "copied_path": [str(p) for p in copied],
    "exists": [p.exists() for p in copied],
    "size_bytes": [p.stat().st_size for p in copied],
}))

print("Versioned transformer artifacts for:", RUN_ID)


,copied_path,exists,size_bytes
0,/Users/veerr_89/Work/projects/up-skin/artifact...,True,7633206
1,/Users/veerr_89/Work/projects/up-skin/artifact...,True,3444736
2,/Users/veerr_89/Work/projects/up-skin/artifact...,True,3444736
3,/Users/veerr_89/Work/projects/up-skin/artifact...,True,39
4,/Users/veerr_89/Work/projects/up-skin/artifact...,True,2760373
5,/Users/veerr_89/Work/projects/up-skin/artifact...,True,3619
6,/Users/veerr_89/Work/projects/up-skin/artifact...,True,48288423
7,/Users/veerr_89/Work/projects/up-skin/artifact...,True,1193


Versioned transformer artifacts for: v002
